### Classificação de grupo etário (young vs older)

Compara **5 modelos** em **3 tarefas** de representação das redes:

1. **EO** — métricas só em olhos abertos  
2. **EC** — métricas só em olhos fechados  
3. **Diff (EC − EO)** — contraste por participante  

Modelos: Dummy (baseline), Logistic Regression L2, SVM RBF, Random Forest, KNN após PCA.

- Alvo: `young` (`20–35`) vs `older` (`55–80`)  
- Features: só métricas de rede (sem idade/gênero — evita vazamento)  
- CV: StratifiedKFold (5), métricas `accuracy`, `balanced_accuracy`, `roc_auc`  
- Amostra analítica: **54 sujeitos** (35 young / 19 older) após o **filtro saudável** do `build_dataset`  
  (metadados LEMON ~227 → intermediários de rede ~158 → processados aqui = filtrados)  
- Resultados **exploratórios** (p ≫ n); LogReg/SVM/RF usam `class_weight="balanced"`  
- Valores `média ± desvio` são a dispersão **entre folds**, não intervalos de confiança

**Features:** métricas de nó alinhadas por eletrodo (`degree_Oz`, `betweenness_Cz`, …; cobertura ≥95% EO&EC). Notebook: `00_redes/03_realign_features_by_electrode.ipynb`.


### Imports


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

### Parâmetros e dados


In [2]:
DATASET_PATH = Path("../../data/processed/dataset_features.parquet")
RANDOM_STATE = 42
N_SPLITS = 5
PCA_COMPONENTS = 10
KNN_NEIGHBORS = 5

YOUNG_AGES = {"20-25", "25-30", "30-35", "35-40"}
OLDER_AGES = {"55-60", "60-65", "65-70", "70-75", "75-80"}

NODE_PREFIXES = (
    "degree_",
    "clustering_",
    "betweenness_",
    "hub_frequency_",
)
GLOBAL_FEATURES = [
    "edges_mean",
    "edges_std",
    "edges_cv",
]

df = pd.read_parquet(DATASET_PATH)
df = df[df["condition"].isin(["EO", "EC"])].copy()
df["age"] = df["age"].astype(str)

duplicates = df.duplicated(subset=["subject_id", "condition"], keep=False)
assert not duplicates.any(), (
    "Existem registros duplicados por participante e condição."
)


def map_age_group(age: str):
    if age in YOUNG_AGES:
        return "young"
    if age in OLDER_AGES:
        return "older"
    return None


df["age_group"] = df["age"].map(map_age_group)
df = df[df["age_group"].notna()].copy()
df["y"] = df["age_group"].map({"young": 0, "older": 1}).astype(int)

print(
    "Amostra analítica (filtro saudável): "
    f"{df['subject_id'].nunique()} sujeitos "
    f"(metadados LEMON ~227; intermediários ~158; processados = filtrados)."
)
print(df.drop_duplicates("subject_id")["age_group"].value_counts())
df[["subject_id", "condition", "age", "age_group", "y"]].head()

Amostra analítica (filtro saudável): 54 sujeitos (metadados LEMON ~227; intermediários ~158; processados = filtrados).
age_group
young    35
older    19
Name: count, dtype: int64


,subject_id,condition,age,age_group,y
0,sub-010069,EC,20-25,young,0
1,sub-010069,EO,20-25,young,0
2,sub-010070,EC,20-25,young,0
3,sub-010070,EO,20-25,young,0
4,sub-010079,EC,25-30,young,0


### Features de rede


In [3]:
FEATURE_COLS = [
    c
    for c in df.columns
    if c.startswith(NODE_PREFIXES) or c in GLOBAL_FEATURES
]
FEATURE_COLS = (
    df[FEATURE_COLS]
    .select_dtypes(include="number")
    .dropna(axis=1, how="all")
    .columns.tolist()
)
print(f"N features de rede: {len(FEATURE_COLS)}")

N features de rede: 215


### Matrizes por tarefa


In [4]:
def build_feature_matrix(frame: pd.DataFrame, condition: str):
    sub = frame[frame["condition"] == condition].copy()
    sub = sub.drop_duplicates("subject_id")
    X = sub[FEATURE_COLS].copy()
    y = sub["y"].copy()
    X.index = sub["subject_id"].values
    y.index = sub["subject_id"].values
    return X, y


def build_diff_matrix(frame: pd.DataFrame):
    eo = (
        frame[frame["condition"] == "EO"]
        .drop_duplicates("subject_id")
        .set_index("subject_id")
    )
    ec = (
        frame[frame["condition"] == "EC"]
        .drop_duplicates("subject_id")
        .set_index("subject_id")
    )
    common = eo.index.intersection(ec.index)
    X = ec.loc[common, FEATURE_COLS] - eo.loc[common, FEATURE_COLS]
    y = eo.loc[common, "y"]
    return X, y


X_eo, y_eo = build_feature_matrix(df, "EO")
X_ec, y_ec = build_feature_matrix(df, "EC")
X_diff, y_diff = build_diff_matrix(df)

tasks = {
    "EO": (X_eo, y_eo),
    "EC": (X_ec, y_ec),
    "Diff_EC_minus_EO": (X_diff, y_diff),
}

for name, (X, y) in tasks.items():
    print(f"{name}: X={X.shape}, y={y.value_counts().to_dict()}")

EO: X=(54, 215), y={0: 35, 1: 19}
EC: X=(54, 215), y={0: 35, 1: 19}
Diff_EC_minus_EO: X=(54, 215), y={0: 35, 1: 19}


### Modelos


In [5]:
def get_models():
    return {
        "Dummy": DummyClassifier(strategy="most_frequent"),
        "LogReg_L2": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(
                        max_iter=5000,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    )),
            ]
        ),
        "SVM_RBF": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "clf",
                    SVC(
                        kernel="rbf",
                        class_weight="balanced",
                        probability=True,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "RandomForest": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                (
                    "clf",
                    RandomForestClassifier(
                        n_estimators=300,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    ),
                ),
            ]
        ),
        "KNN_PCA": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)),
                ("clf", KNeighborsClassifier(n_neighbors=KNN_NEIGHBORS)),
            ]
        ),
    }


models = get_models()
list(models)

['Dummy', 'LogReg_L2', 'SVM_RBF', 'RandomForest', 'KNN_PCA']

### Avaliação (CV estratificada)


In [6]:
def evaluate_all(X, y, models, task: str) -> pd.DataFrame:
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    scoring = {
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "roc_auc": "roc_auc",
    }
    rows = []
    for name, model in models.items():
        scores = cross_validate(model, X, y, cv=cv, scoring=scoring)
        row = {
            "task": task,
            "model": name,
            "n": len(y),
            "accuracy_mean": float(np.mean(scores["test_accuracy"])),
            "accuracy_std": float(np.std(scores["test_accuracy"])),
            "balanced_accuracy_mean": float(np.mean(scores["test_balanced_accuracy"])),
            "balanced_accuracy_std": float(np.std(scores["test_balanced_accuracy"])),
            "roc_auc_mean": float(np.mean(scores["test_roc_auc"])),
            "roc_auc_std": float(np.std(scores["test_roc_auc"])),
        }
        rows.append(row)
        print(
            f"[{task}] {name}: "
            f"acc={row['accuracy_mean']:.3f}±{row['accuracy_std']:.3f} | "
            f"bal={row['balanced_accuracy_mean']:.3f}±{row['balanced_accuracy_std']:.3f} | "
            f"auc={row['roc_auc_mean']:.3f}±{row['roc_auc_std']:.3f}"
        )
    return pd.DataFrame(rows)


results = [evaluate_all(X, y, models, task) for task, (X, y) in tasks.items()]
df_results = pd.concat(results, ignore_index=True)
df_results

[EO] Dummy: acc=0.649±0.025 | bal=0.500±0.000 | auc=0.500±0.000
[EO] LogReg_L2: acc=0.722±0.116 | bal=0.692±0.108 | auc=0.793±0.086
[EO] SVM_RBF: acc=0.724±0.096 | bal=0.713±0.074 | auc=0.788±0.072
[EO] RandomForest: acc=0.778±0.042 | bal=0.745±0.057 | auc=0.821±0.023
[EO] KNN_PCA: acc=0.702±0.096 | bal=0.677±0.090 | auc=0.730±0.099
[EC] Dummy: acc=0.649±0.025 | bal=0.500±0.000 | auc=0.500±0.000
[EC] LogReg_L2: acc=0.691±0.178 | bal=0.671±0.177 | auc=0.686±0.162
[EC] SVM_RBF: acc=0.629±0.129 | bal=0.633±0.143 | auc=0.702±0.155
[EC] RandomForest: acc=0.629±0.116 | bal=0.550±0.124 | auc=0.714±0.213
[EC] KNN_PCA: acc=0.687±0.166 | bal=0.655±0.159 | auc=0.729±0.152
[Diff_EC_minus_EO] Dummy: acc=0.649±0.025 | bal=0.500±0.000 | auc=0.500±0.000
[Diff_EC_minus_EO] LogReg_L2: acc=0.613±0.130 | bal=0.555±0.115 | auc=0.536±0.096
[Diff_EC_minus_EO] SVM_RBF: acc=0.725±0.112 | bal=0.660±0.113 | auc=0.679±0.124
[Diff_EC_minus_EO] RandomForest: acc=0.687±0.133 | bal=0.620±0.140 | auc=0.719±0.140
[Diff

,task,model,n,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std
0,EO,Dummy,54,0.649091,0.025455,0.500000,0.000000,0.500000,0.000000
1,EO,LogReg_L2,54,0.721818,0.115508,0.691667,0.107960,0.792857,0.085714
2,EO,SVM_RBF,54,0.723636,0.096484,0.713095,0.074097,0.788095,0.071587
3,EO,RandomForest,54,0.778182,0.042094,0.745238,0.056519,0.821429,0.022588
4,EO,KNN_PCA,54,0.701818,0.095934,0.677381,0.089674,0.729762,0.098788
5,EC,Dummy,54,0.649091,0.025455,0.500000,0.000000,0.500000,0.000000
6,EC,LogReg_L2,54,0.690909,0.178145,0.671429,0.177353,0.685714,0.161940
7,EC,SVM_RBF,54,0.629091,0.129385,0.633333,0.143085,0.702381,0.155036
8,EC,RandomForest,54,0.629091,0.115908,0.550000,0.124438,0.714286,0.213092
9,EC,KNN_PCA,54,0.687273,0.166003,0.654762,0.158829,0.728571,0.151691


### Tabela resumo


In [7]:
summary = df_results.copy()
summary["accuracy"] = summary.apply(
    lambda r: f"{r['accuracy_mean']:.3f} ± {r['accuracy_std']:.3f}", axis=1
)
summary["balanced_accuracy"] = summary.apply(
    lambda r: f"{r['balanced_accuracy_mean']:.3f} ± {r['balanced_accuracy_std']:.3f}",
    axis=1,
)
summary["roc_auc"] = summary.apply(
    lambda r: f"{r['roc_auc_mean']:.3f} ± {r['roc_auc_std']:.3f}", axis=1
)
summary_pivot = summary.pivot(
    index="model", columns="task", values="balanced_accuracy"
)
summary_pivot

task,Diff_EC_minus_EO,EC,EO
model,,,
Dummy,0.500 ± 0.000,0.500 ± 0.000,0.500 ± 0.000
KNN_PCA,0.561 ± 0.056,0.655 ± 0.159,0.677 ± 0.090
LogReg_L2,0.555 ± 0.115,0.671 ± 0.177,0.692 ± 0.108
RandomForest,0.620 ± 0.140,0.550 ± 0.124,0.745 ± 0.057
SVM_RBF,0.660 ± 0.113,0.633 ± 0.143,0.713 ± 0.074


In [8]:
fig = px.bar(
    df_results,
    x="model",
    y="balanced_accuracy_mean",
    error_y="balanced_accuracy_std",
    color="task",
    barmode="group",
    title="Balanced accuracy por modelo e tarefa (CV 5-fold)",
    labels={
        "balanced_accuracy_mean": "Balanced accuracy",
        "model": "Modelo",
        "task": "Tarefa",
    },
)
fig.update_layout(yaxis_range=[0, 1], template="plotly_white", height=450)
fig.show()

### Interpretação (guia)

- **Dummy (`most_frequent`)** prediz sempre a classe majoritária (`young` ≈ 65% dos sujeitos). Accuracy alta no Dummy **não** indica sinal; use **balanced accuracy** e **ROC-AUC**.
- Comparar cada modelo ao Dummy na mesma tarefa: só há evidência útil se balanced accuracy / AUC forem consistentemente superiores.
- **EO vs EC vs Diff**: se Diff for melhor, o contraste de estado carrega informação etária além do nível absoluto; se EO ou EC forem melhores, o estado isolado já basta.
- `média ± desvio` = dispersão **entre os 5 folds** (não é intervalo de confiança).
- Funil da amostra: metadados (~227) → intermediários de rede (~158) → processados após filtro saudável (**54**). Ampliar a amostra (ou revisitar critérios de inclusão) é o passo mais relevante para confiabilidade.
- n = 54 e CV 5-fold → resultados exploratórios (sem grid search).

Permutação EO: `08_ml_age_permutation_eo.ipynb`.

Próximo passo (permutação EO, LogReg/SVM): `08_ml_age_permutation_eo.ipynb`.

Robustez / interpretabilidade (EO + SVM): `09_ml_age_robustness_interpretability.ipynb`.


### Amostra

Funil da amostra: metadados (227) → participantes com redes processadas (158) → amostra analítica após os critérios de inclusão saudável (54). A documentação dos critérios de inclusão e a avaliação de sua influência sobre a composição etária da amostra são necessárias para interpretar a capacidade de generalização dos resultados.

Na amostra analítica de 54 participantes, os modelos treinados com métricas das redes na condição de olhos abertos apresentaram médias superiores às do classificador de referência. A Regressão Logística alcançou acurácia balanceada de 0,692 ± 0,108 e ROC-AUC de 0,793 ± 0,086, e o SVM, acurácia balanceada de 0,713 ± 0,074 e ROC-AUC de 0,788 ± 0,072. O Random Forest teve o melhor desempenho em EO (acurácia balanceada de 0,745 ± 0,057; ROC-AUC de 0,821 ± 0,023). Os resultados obtidos na condição de olhos fechados foram mais modestos e apresentaram maior variabilidade entre os folds. As diferenças entre as métricas de EC e EO não demonstraram capacidade discriminativa clara. Esses achados sugerem, de forma exploratória, que as redes na condição de olhos abertos podem conter informações associadas ao grupo etário. Entretanto, o número reduzido de participantes e a elevada quantidade de características exigem validação adicional.